In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json 

In [2]:
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

### Vector Store Creation

In [39]:
vector_store = client.vector_stores.create(name="Industrial_taxonomy_space")
taxonomy_vector_id=vector_store.id
print(taxonomy_vector_id)

vs_6914573daa5c81918616d8a8da7351d4


### Uploading to vector store

In [ ]:
vector_store_id=taxonomy_vector_id

# Directory containing the split files
split_dir = "split_taxonomy"

# Loop through each file in the directory and upload
for filename in os.listdir(split_dir):
    if filename.endswith(".json"):  # Only upload JSON files
        file_path = os.path.join(split_dir, filename)
        print(f"Uploading {filename}...")
        
        # Open the file and upload
        with open(file_path, "rb") as f:
            file_response = client.vector_stores.files.upload_and_poll(
                vector_store_id=vector_store_id,
                file=f
            )
        
        # Check status (optional, for debugging)
        print(f"Uploaded {filename}: Status - {file_response.status}")
        
print("All files uploaded successfully!")

In [29]:
article_text="""HCA Healthcare is one of the leading hospital chains in the U.S., with a significant presence across the country, although its business is concentrated primarily in Texas and Florida.\n\nThe industry is competitive, but HCA consistently performs well nonetheless. The company has several competitive advantages, including its extensive network of facilities that span acute care hospitals, specialty care, urgent care, and more. This diversified mix of facilities enables it to retain patients within its ecosystem, granting it significant negotiating power with private insurers and government payers.\n\nAs a result, HCA Healthcare has built relationships with these third-party institutions. Further, HCA routinely invests in cutting-edge technology, another factor that allows it to attract patients. These dynamics have enabled the company to increase its market share over time. It had a 24% market share in 2012, and 10 years later, that had risen to 27%.\n\nHCA also records consistent revenue and profits. In the second quarter, the company's top line reached $18.6 billion, representing a 6.4% increase compared to the same period last year. The company's earnings per share climbed 23.5% year over year to $6.83.\n\nOver the next decade, HCA Healthcare should benefit from increased hospital spending. In the meantime, the company will continue to implement the same strategy that has made it so successful over the years. It's also worth noting that there are significant regulatory and financial barriers to entry in the industry.\n\nNewcomers then have to compete with established corporations to attract patients and third-party payers, but it's challenging to do so because those, like HCA Healthcare, that have been around for a while have already built solid relationships with major players in the healthcare sector. That's why it's hard to imagine HCA Healthcare completely losing its position. The stock should continue performing well over the long run."""

### Retrieval from Vector store

In [30]:
results = client.vector_stores.search(
    vector_store_id="vs_6914573daa5c81918616d8a8da7351d4",
    query=article_text,
)

for item in results.data:
    print(f"Score: {item.score:.3f}")
    print(item.content[0].text[:300], "...\n")


Score: 0.863
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Digital Health",
  "Description": "Software-enabled care delivery, monitoring, and patient engagement."
} ...

Score: 0.861
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Health Insurance",
  "Description": "Health risk pooling, provider networks, and claims processing."
} ...

Score: 0.861
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Medical Devices",
  "Description": "Diagnostic and therapeutic devices for hospital and home use."
} ...

Score: 0.854
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Remote Patient Monitoring (RPM)",
  "Description": "Connected devices and platforms tracking patients at home."
} ...

Score: 0.851
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Telemedicine",
  "Description": "Virtual consultations and remote care platforms."
} ...

Score: 0.849
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Clinical Trials",
  "Description":

In [31]:
results_sorted = sorted(results.data, key=lambda x: x.score, reverse=True)

### Formatting Vector Retrieval 

In [32]:

def format_results(results):
    """
    Takes vector store search results and returns a formatted string
    containing only Industry and Sub-category, one per line, suitable
    to feed into the LLM.
    """
    formatted_lines = []

    for item in results:
        for part in item.content:
            try:
                data = json.loads(part.text)  
                industry = data.get("Industry")
                sub_category = data.get("Sub-category")
                if industry and sub_category:
                    # Format exactly like your candidates_str example
                    formatted_lines.append(f"- {industry} (Sub-category: {sub_category})")
            except json.JSONDecodeError:
                continue

    # Join all lines into a single string
    return "\n".join(formatted_lines)


formatted_results=format_results(results_sorted)

print("Candidates to feed into LLM:\n", formatted_results)


Candidates to feed into LLM:
 - Healthcare & Biotech (Sub-category: Digital Health)
- Healthcare & Biotech (Sub-category: Health Insurance)
- Healthcare & Biotech (Sub-category: Medical Devices)
- Healthcare & Biotech (Sub-category: Remote Patient Monitoring (RPM))
- Healthcare & Biotech (Sub-category: Telemedicine)
- Healthcare & Biotech (Sub-category: Clinical Trials)
- Healthcare & Biotech (Sub-category: Gene Therapy & Genomics)
- Healthcare & Biotech (Sub-category: Pharmaceuticals)
- Healthcare & Biotech (Sub-category: Biomanufacturing / Bio-Production)
- Healthcare & Biotech (Sub-category: AI Diagnostics)


### System Prompt

In [33]:
SYSTEM_PROMPT = f"""You are an industrial taxonomy matcher.
 Your job: Read the news article carefully and shortlist only the CENTRAL / EMPHASIZED industries and CENTRAL / EMPHASIZED sub-categories from the provided taxonomy evidence that is highly relevent to the article context.
 Rules & Best Practices:
 1) Primary soure of output is from the provided taxonomy evidence.
    - Prefer the exact strings  for both Main Industry and Sub-category.

 2) What counts as " Central / Emphasized " :
    - The article explicitly covers the topic, a company from that industry, a product, major policy/market movement, or cause effect impacts on that industry.
    - Background mentions without substance DO NOT qualify.


 3) Relevance rubric (pick ONE label):
   - HIGH: Central theme; multiple paragraphs or the thesis of the article revolves around it; material implications.
   - MEDIUM: Substantive but secondary; a clear thread or section is about it; non-trivial impact.
   - LOW: Brief yet meaningful; not central but clearly connected with a tangible implication.
   - If NONE of these thresholds are met, DO NOT include the industry at all.

4)  Reason rubric:
   - Keep it short and concise.

 5) Disambiguation & Mapping:
    - Map synonyms/aliases to the closest taxonomy nodes (e.g., “EVs” → Automotive & Mobility → Electric Vehicles (EV);
     “lithium/battery metals” → Materials & Mining → Metals & Mining OR Rare Earths (when magnets/REEs are discussed);
     “grid-scale batteries” → Energy & Renewables → Battery Storage).
    - If retrieval yields nothing relevant, you may add at most one inferred item  as "emergent_industry_candidate".

 7) Hallucination Guardrails
   - Use only information available in the article or from retrieved taxonomy evidence . If uncertain, omit.
   - Do NOT guess, paraphrase, or extrapolate.  
   - Do NOT be creative or flexible in interpretation.
"""

### User Prompt

In [38]:
USER_PROMPT=f"""
Article:
{article_text}

Retrieved Taxonomy evidence:{formatted_results}

Task:
- Shortlist only CENTRAL / EMPHASIZED industries and  CENTRAL / EMPHASIZED sub-categories present in the retrieved evidence that meet a LOW-or-above relevance.
- For each shortlisted item return: industry, sub_category, relevance (LOW/MEDIUM/HIGH), reason.
  - If the item is taken from the evidence, the reason MUST end with Retrieved from vector store.
  - If you infer one emergent item (allowed only if nothing relevant exists in evidence), mark it under emergent_industry_candidate and the reason should end with "Inferred".
- Answer deterministically , don't give different outputs for the same input.


"""

### JSON Schema

In [39]:
textt_format={
            "format": {
                "type": "json_schema",
                "name": "IndustryClassification",
                "schema": {
                    "type": "object",
                    "properties": {
                        "shortlisted_industries": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "sector": { "type": "string" },
                                    "sub_category": { "type": "string" },
                                    "relevance": {
                                        "type": "string",
                                        "enum": ["LOW", "MEDIUM", "HIGH"]
                                    },
                                    "reason": { "type": "string", "maxLength":220 }
                                },
                                "required": ["sector", "sub_category", "relevance", "reason"],
                                "additionalProperties": False
                            }
                        },
                        "emergent_industry_candidate": {
                            "anyOf": [
                                {
                                    "type": "object",
                                    "properties": {
                                        "label": { "type": "string" },
                                        "rationale": { "type": "string" }
                                    },
                                    "required": ["label", "rationale"],
                                    "additionalProperties": False
                                },
                                { "type": "null" }
                            ]
                        }
                    },
                    "required": ["shortlisted_industries", "emergent_industry_candidate"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }

### LLM Call

In [42]:
response = client.responses.create(
        model="gpt-5.1",        
        input=[
            {"role": "system", "content": SYSTEM_PROMPT },
            {"role": "user", "content": USER_PROMPT}
        ],
        text=textt_format,
                           
)
result=response.output_text
print(result)  

{"shortlisted_industries":[],"emergent_industry_candidate":{"label":"Healthcare & Biotech (Hospitals & Healthcare Facilities)","rationale":"The article is centrally about HCA Healthcare’s hospital and care-facility operations, but no matching hospitals/facilities sub-category exists in the retrieved evidence. Inferred"}}


In [43]:
try:
    parsed = json.loads(result)
   
    pretty_json = json.dumps(parsed, indent=4, ensure_ascii=False)

    print(pretty_json)

except json.JSONDecodeError as e:
    print("Invalid JSON:", e)

{
    "shortlisted_industries": [],
    "emergent_industry_candidate": {
        "label": "Healthcare & Biotech (Hospitals & Healthcare Facilities)",
        "rationale": "The article is centrally about HCA Healthcare’s hospital and care-facility operations, but no matching hospitals/facilities sub-category exists in the retrieved evidence. Inferred"
    }
}
